# 《LangChain 大模型集成实验》

## 一、实验目的
1. 掌握 LangChain 基础使用，理解 ChatModel 调用规范
2. 掌握 .env 环境变量管理密钥的开发规范
3. 掌握两种模型调用方式：框架封装调用、原生SDK调用
4. 熟悉大模型流式输出

## 二、实验环境
- 系统：Windows 10
- Python版本：3.10
- 虚拟环境：Miniconda
- 开发工具：VS Code、Jupyter Notebook
- 依赖库：langchain、langchain-deepseek、langchain-community、python-dotenv、openai
- 模型：DeepSeek、阿里通义千问

## 三、实验原理

### 3.1 LangChain语言模型介绍

一个 AI 应用的核心就是它所依赖的大语言模型，LangChain作为一个“工具”，不提供任何 LLMs，而是依赖于第三方集成各种大模型。比如，将 OpenAI、Anthropic、Hugging Face 、LlaMA、阿里Qwen、ChatGLM等平台的模型无缝接入到你的应用。

LangChain 模型接口可参考官方文档：https://reference.langchain.com/python/langchain_core/language_models/

### 3.2 LangChain模型分类

LangChain中将大语言模型分为以下几种，我们主要使用的是聊天模型：
![alt text](img/image1.png)

### 3.3 ChatModel主要参数
![alt text](img/image2.png)

以上的标准参数，也只是适用于部分的大语言模型，有些参数在特定模型中可能是无效的，这些标准化参数仅对 LangChain 官方提供集成包的模型（如 langchain-openai、langchain-anthropic）生效，在langchain-community包中的第三方模型，则不需要遵守这些标准化参数的规则。

### 3.4 Message组件

调用模型后返回了一条AI消息，在LangChain中，消息有几种不同的类型。所有消息都有 type 、 content 、 response_metadata 等属性。

下面是这几个属性的作用：
![alt text](img/image3.png)

## 四、实验内容

### 4.1 安装依赖库

在Jupyter中使用 `!` 执行系统命令安装所需的依赖包。

In [ ]:
%pip install langchain==1.2.18 langchain-deepseek==1.0.1 langchain-community==0.4.1 python-dotenv==1.2.2

### 4.2 配置环境变量

在同级目录新建 `.env` 文件，写入密钥：
```
DEEPSEEK_API_KEY=xxx
QWEN_API_KEY=xxx
```

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
qwen_api_key = os.getenv("QWEN_API_KEY")

### 4.3 LangChain封装调用DeepSeek

使用官方封装包快速调用对话大模型。

In [2]:
from langchain_deepseek import ChatDeepSeek

llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    api_key=deepseek_api_key,
)

res = llm.invoke("什么是LangChain？")
print(res)

content='这是一个关于 **LangChain** 的详细解释。\n\n简单来说，**LangChain 是一个用于开发由大语言模型（LLM）驱动的应用程序的框架**。\n\n你可以把它想象成 **LLM 应用的“乐高积木”**。它提供了一套标准化的工具和组件，让开发者能够轻松地将不同的 AI 模型、数据源、工具和逻辑流程组合在一起，构建出强大的、复杂的 AI 应用，而无需从零开始编写所有底层代码。\n\n---\n\n### 核心思想：解决 LLM 的局限性\n\nLangChain 的出现是为了解决大语言模型本身的一些关键局限性：\n\n1.  **知识过时**：LLM 的训练数据有截止日期，无法知道最新的信息。\n2.  **无法访问私有数据**：LLM 不知道你公司的内部文档、数据库或个人笔记。\n3.  **缺乏“记忆”**：在对话中，LLM 默认不记得你之前说过什么（上下文窗口有限）。\n4.  **无法执行操作**：LLM 只能生成文本，不能直接调用 API、发送邮件、查询数据库或执行代码。\n\nLangChain 通过提供一系列抽象和工具，完美地解决了这些问题。\n\n---\n\n### LangChain 的核心组件\n\nLangChain 主要由以下几个关键部分组成：\n\n#### 1. 模型（Models）\n这是最基础的部分，即你使用的 LLM。LangChain 提供了一个统一的接口，让你可以轻松切换不同的模型提供商，例如：\n\n-   **OpenAI** (GPT-3.5, GPT-4)\n-   **Google** (Gemini, PaLM)\n-   **Anthropic** (Claude)\n-   **开源模型** (Llama 2, Mistral, Falcon) 通过 Hugging Face 或 Ollama 等平台接入。\n\n#### 2. 提示词模板（Prompts）\n管理、优化和动态构建发送给 LLM 的提示词。例如，你可以创建一个模板：“请用{语言}将以下内容翻译成{目标语言}：{文本}”。然后，你只需传入变量 `语言`、`目标语言` 和 `文本`，LangChain 就会自动生成完整的提示词。\n\n#### 3. 索引（Indexes） / 检索增强生成（RAG）\n这是 Lan

### 4.4 原生SDK调用通义千问（流式输出）

调用阿里百炼接口，开启思考链，流式返回数据。

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=qwen_api_key,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

completion = client.chat.completions.create(
    model="qwen3.6-plus",
    messages=[{"role":"user","content":"你是谁"}],
    extra_body={"enable_thinking": True},
    stream=True
)

is_answering = False
print("\n=====思考=====")

for chunk in completion:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta

    if hasattr(delta,"reasoning_content") and delta.reasoning_content:
        print(delta.reasoning_content,end="")
    if hasattr(delta,"content") and delta.content:
        if not is_answering:
            print("\n=====回复=====")
            is_answering = True
        print(delta.content,end="")

## 五、实验总结

1. 完成 langchain、langchain-deepseek、langchain-community、python-dotenv 等核心依赖的安装，搭建了基础开发环境
2. 通过 .env 文件管理 API Key，避免密钥硬编码，养成了安全开发的习惯
3. 使用 ChatDeepSeek 封装调用 DeepSeek 模型，体验了 LangChain 统一接口的便捷性
4. 使用原生 OpenAI SDK 调用通义千问，理解了封装调用与原生SDK两种方式各自的适用场景
5. 实现了流式输出，通过  逐片获取内容，掌握了处理空 chunk 的容错写法